# Chatbot GPT from scratch

## Model

In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, IterableDataset
import torch.nn.init as init
from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
torch.device("cuda" if torch.cuda.is_available() else "cpu")

device(type='cuda')

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super().__init__()
        pe = torch.zeros((max_seq_length, d_model))
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]
    

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "Vetor de embedding precisa ser divisivel pelo número de cabeças da camada de atenção!"
        self.head_dim = d_model // num_heads
        self.d_model, self.num_heads = d_model, num_heads
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.output_linear = nn.Linear(d_model, d_model)

    def split_heads(self, x, encoder_output=None):
        # Entra Q, K, V com dimensão (batch_size, sequence_length, d_model)
        # Reshape para (batch_size, sequence_length, num_heads, d_model)
        # Reordering para (batch_size, num_heads, sequence_length, d_model)
        if encoder_output is None:
            x = torch.reshape(x, shape=(x.shape[0], x.shape[1], self.num_heads, self.head_dim)) #.contiguous()
            x = x.permute(0, 2, 1, 3)
        else:
            raise NotImplementedError("Modelo ainda não compatível com Encoder.")
        return x

    def compute_attention_scores(self, q_linear_out, k_linear_out, v_linear_out, mask=None):
        qk_dot_product = torch.matmul(q_linear_out, k_linear_out.transpose(2, 3)) / self.head_dim ** 0.5

        if mask is not None:
            qk_dot_product = qk_dot_product.masked_fill(mask == 0, float('-inf'))

        attn_scores = nn.functional.softmax(qk_dot_product, dim=-1)
        attn_weighted_v = torch.matmul(attn_scores, v_linear_out)

        return attn_weighted_v


    def combine_heads(self, x):
        x = x.permute(0, 2, 1, 3).contiguous()
        return torch.reshape(x, shape=(x.shape[0], x.shape[1], int(x.shape[2] * x.shape[3])))

    def forward(self, x, mask):
        q_linear_out = self.split_heads(self.q(x))
        k_linear_out = self.split_heads(self.k(x))
        v_linear_out = self.split_heads(self.v(x))
        
        attn_weighted_v = self.compute_attention_scores(q_linear_out, k_linear_out, v_linear_out, mask=mask)
        attn_weighted_v = self.combine_heads(attn_weighted_v)
        return self.output_linear(attn_weighted_v)


class FeedForwardSubLayer(nn.Module):
    def __init__(self, d_model, hidden_size):
        super().__init__()
        self.ff_1 = nn.Linear(d_model, hidden_size)
        self.ff_2 = nn.Linear(hidden_size, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.ff_2(self.relu(self.ff_1(x)))
    

class DecoderBlock(nn.Module):
    def __init__(self, d_model, hidden_size, num_heads, dropout=0.1):
        super().__init__()
        self.feed_forward = FeedForwardSubLayer(d_model, hidden_size)
        self.mha = MultiHeadAttention(d_model, num_heads) # nn.MultiheadAttention()
        self.norm_1 = nn.LayerNorm(d_model)
        self.norm_2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, tgt_mask):
        x = self.norm_1(x + self.dropout(self.mha(x, mask=tgt_mask)))
        x = self.norm_2(x + self.dropout(self.feed_forward(x)))
        return x
    

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, max_sequence_length, n_layers, hidden_size, num_heads, dropout=0.1):
        super(TransformerDecoder, self).__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model, padding_idx=0)
        self.pe = PositionalEncoding(d_model, max_sequence_length)
        self.layers = nn.ModuleList(
            [DecoderBlock(d_model, hidden_size, num_heads, dropout) for _ in range(n_layers)]
        )
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, x, tgt_mask):
        x = self.embedding(x)
        x = self.pe(x)
        for layer in self.layers:
            x = layer(x, tgt_mask)
        out = self.output_layer(x)
        return out


## Dataset Prep

### Tokenizer Char Level

In [4]:
class TokenizerChar:
    def __init__(self):
        self.chr_to_idx = {chr(v): v for v in range(1, 257)}
        self.chr_to_idx['<SOS>'] = 257
        self.chr_to_idx['<EOS>'] = 258
        self.chr_to_idx['<PAD>'] = 0
        self.chr_to_idx['<UNK>'] = 259
        self.chr_to_idx['<EOP>'] = 260
        self.special_tokens_chrs = ['<SOS>', '<EOS>', '<PAD>', '<UNK>', '<EOP>']

        self.idx_to_chr = {v: k for k, v in self.chr_to_idx.items()}

        self.vocab_size = len(self.chr_to_idx.keys())

    def encode(self, char):
        if char in self.chr_to_idx.keys():
            return self.chr_to_idx[char]
        else:
            return 259
        
    def encode_text(self, text):
        tokens = []
        i = 0
        while i < len(text):
            # Detecta tokens especiais (todos têm 5 caracteres e começam com '<')
            if len(text) - i >= 5:
                if text[i] == '<' and i + 4 < len(text) and text[i:i+5] in self.special_tokens_chrs:
                    tokens.append(self.encode(text[i:i+5]))
                    i += 5
                else:
                    tokens.append(self.encode(text[i]))
                    i += 1
            else:
                tokens.append(self.encode(text[i]))
                i += 1
        return tokens
    
    def decode(self, token_idx):
        return self.idx_to_chr[token_idx]
    
    def sos_token(self):
        return '<SOS>'
    
    def sos_token_idx(self):
        return self.chr_to_idx['<SOS>']

    def eos_token(self):
        return '<EOS>'
    
    def eos_token_idx(self):
        return self.chr_to_idx['<EOS>']
    
    def pad_token(self):
        return '<PAD>'
    
    def pad_token_idx(self):
        return self.chr_to_idx['<PAD>']
    
    def eop_token(self):
        return '<EOP>'
    
    def eop_token_idx(self):
        return self.chr_to_idx['<EOP>']
    
    def get_vocab_size(self):
        return self.vocab_size

### Tokenizer Byte Pair

In [5]:
from collections import defaultdict
from typing import List


class TrieText:
    def __init__(self, init_token_idx=5):
        self.trie = defaultdict(dict)
        self.keys = set()
        self.token_to_idx = defaultdict(lambda: None)
        self.token_idxs = [init_token_idx]
    
    def get_trie(self):
        return self.trie
    
    def add_node(self, parent_nodes: List[str], new_node: str):
        trie = self.trie

        if parent_nodes == []:
            trie[new_node] = defaultdict(dict)
            self.keys.add(new_node)
            return
        
        last_node = ''
        for node in parent_nodes:
            trie = trie[node]
            last_node = node

        if last_node in new_node:
            trie[new_node] = defaultdict(dict)
            self.keys.add(new_node)
        else:
            raise ValueError("New node is not valid! It must contain previous parent node string values.")
    
    def get_node_path(self, node: str, get_parent_nodes=False):
        nodes = []
        current_node = self.trie.copy()
        for i in range(1, len(node) + 1):
            key = node[:i]
            if key in current_node.keys():
                nodes.append(key)
                current_node = current_node[key]
        if get_parent_nodes and node in self.keys:
            return nodes[:-1]
        return nodes
    
    def get_child_nodes(self, parent_node: str):
        paths = []
        longest_text = parent_node

        def dfs(trie_subtree, path):
            nonlocal longest_text
            node = path[-1]
            if len(node) > len(longest_text):
                longest_text = node
            children = list(trie_subtree.keys())
            if not children:
                paths.append(path.copy())
            else:
                for child in children:
                    dfs(trie_subtree[child], path + [child])

        # Verifica se parent_node existe na trie
        if parent_node not in self.trie:
            return [], parent_node

        dfs(self.trie[parent_node], [parent_node])
        return paths, longest_text
    
    def get_token_idx(self, token: str):
        return self.token_to_idx[token]
    
    def mount_trie(self, str_set: set):
        assert isinstance(str_set, set), "Input must be a set of strings!"
        self.keys = set()
        vocab = sorted(list(str_set))
        print(vocab)
        for str_ in vocab:
            parent_nodes_path = self.get_node_path(str_, get_parent_nodes=True)
            self.add_node(parent_nodes=parent_nodes_path, new_node=str_)
        self.assign_token_idxs()

    def assign_token_idxs(self):
        for key in sorted(list(self.keys)):
            current_t_idx = self.token_idxs[-1]
            self.token_to_idx[key] = current_t_idx
            self.token_idxs.append(current_t_idx + 1)

In [6]:
trie = TrieText()

In [7]:
# set_ = {'car', 'carro', 'cars', 'bananas', 'bana', 'banana', 'b'}
set_ = {'a', 'b', 'd', 'e', 'f', 'n', 'r', 't', ' ', 'd ', 'ed ', 'fr', 'fred ', 'fra'}

In [8]:
trie.mount_trie(set_)

[' ', 'a', 'b', 'd', 'd ', 'e', 'ed ', 'f', 'fr', 'fra', 'fred ', 'n', 'r', 't']


In [9]:
print(trie.get_token_idx('bananas'))

None


In [10]:
trie.token_to_idx

defaultdict(<function __main__.TrieText.__init__.<locals>.<lambda>()>,
            {' ': 5,
             'a': 6,
             'b': 7,
             'd': 8,
             'd ': 9,
             'e': 10,
             'ed ': 11,
             'f': 12,
             'fr': 13,
             'fra': 14,
             'fred ': 15,
             'n': 16,
             'r': 17,
             't': 18,
             'bananas': None})

In [11]:
trie.get_node_path("fra")

['f', 'fr', 'fra']

In [12]:
trie.get_trie()

defaultdict(dict,
            {' ': defaultdict(dict, {}),
             'a': defaultdict(dict, {}),
             'b': defaultdict(dict, {}),
             'd': defaultdict(dict, {'d ': defaultdict(dict, {})}),
             'e': defaultdict(dict, {'ed ': defaultdict(dict, {})}),
             'f': defaultdict(dict,
                         {'fr': defaultdict(dict,
                                      {'fra': defaultdict(dict, {}),
                                       'fred ': defaultdict(dict, {})})}),
             'n': defaultdict(dict, {}),
             'r': defaultdict(dict, {}),
             't': defaultdict(dict, {})})

In [13]:
trie.get_child_nodes("f")

([['f', 'fr', 'fra'], ['f', 'fr', 'fred ']], 'fred ')

In [14]:
trie.add_node([], "a")

In [15]:
trie.get_node_path("a", get_parent_nodes=False)

['a']

In [16]:
trie.add_node(["a"], "ab")

In [17]:
trie.get_node_path("ab")

['a', 'ab']

In [18]:
# trie.add_node(["a"], "bc")

In [19]:
trie.get_node_path("abelha")

['a', 'ab']

In [20]:
print(trie.get_trie())

defaultdict(<class 'dict'>, {' ': defaultdict(<class 'dict'>, {}), 'a': defaultdict(<class 'dict'>, {'ab': defaultdict(<class 'dict'>, {})}), 'b': defaultdict(<class 'dict'>, {}), 'd': defaultdict(<class 'dict'>, {'d ': defaultdict(<class 'dict'>, {})}), 'e': defaultdict(<class 'dict'>, {'ed ': defaultdict(<class 'dict'>, {})}), 'f': defaultdict(<class 'dict'>, {'fr': defaultdict(<class 'dict'>, {'fra': defaultdict(<class 'dict'>, {}), 'fred ': defaultdict(<class 'dict'>, {})})}), 'n': defaultdict(<class 'dict'>, {}), 'r': defaultdict(<class 'dict'>, {}), 't': defaultdict(<class 'dict'>, {})})


In [21]:
trie.get_node_path("abelhas")

['a', 'ab']

In [22]:
trie.get_node_path("a")

['a']

In [23]:
trie.add_node(['a', 'ab'], "abelha")

In [24]:
trie.get_node_path("abelha")

['a', 'ab', 'abelha']

In [25]:
trie.get_node_path("abel")

['a', 'ab']

In [26]:
trie.add_node(['a', 'ab'], "abel")

In [27]:
trie.get_node_path("abel")

['a', 'ab', 'abel']

In [28]:
trie.get_node_path("abelha")

['a', 'ab', 'abel']

In [29]:
trie.get_node_path("abelhas")

['a', 'ab', 'abel']

In [30]:
class TokenizerBPE:
    def __init__(self, vocab_size=50_000):
        self.trie = TrieText()
        self.token_to_idx = dict()
        self.token_to_idx['<PAD>'] = 0
        self.token_to_idx['<SOS>'] = 1
        self.token_to_idx['<EOS>'] = 2
        self.token_to_idx['<UNK>'] = 3
        self.token_to_idx['<EOP>'] = 4
        self.special_tokens = ['<SOS>', '<EOS>', '<PAD>', '<UNK>', '<EOP>']
        self.trie.token_to_idx = self.token_to_idx
        self.idx_to_token = {v: k for k, v in self.token_to_idx.items()}
        self.vocab_size = vocab_size

    def fit(self, text_corpus, max_bpe_iter=10_000):
        print("Iniciando treinamento do TokenizerBPE...")

        print("Detectando e removendo tokens especiais...")
        for i in tqdm(range(0, len(text_corpus) - 4), total=(len(text_corpus) - 4)):
            str_ = text_corpus[i:i+5]
            if self.detect_special_token(str_):
                text_corpus = text_corpus.replace(str_, "")

        print("Removendo identificadores de tokens especiais...")
        text_corpus = text_corpus.replace("<", "")
        text_corpus = text_corpus.replace(">", "")
        
        print("Treinando BPE...")
        text_symbols = list(text_corpus)
        bpe_iter = -1
        for i in tqdm(range(max_bpe_iter)):
            # if i % 2 == 0:
            #     print(f"Tamanho atual do vocabulário: {len(set(text_symbols))}")
            # print(f"{text_symbols=}")
            bpe_iter += 1
            freqs = dict()
            for i in range(0, len(text_symbols) - 1):
                adjacent_symbols = text_symbols[i:i+2]
                adjacent_symbols_str = "".join(adjacent_symbols)
                if adjacent_symbols_str not in freqs.keys():
                    freqs[adjacent_symbols_str] = 0
                freqs[adjacent_symbols_str] += 1
            # print(freqs)

            max_freq = 1
            merged_symbol = ""
            for pair, freq in freqs.items():
                if freq > max_freq:
                    merged_symbol = pair
                    max_freq = freq
            # print(f"{merged_symbol=} - {max_freq=}")
            if merged_symbol == "":
                print("Não existem símbolos possíveis de serem mergeados. Tokenizer treinado.")
                break
            
            new_text_symbols = []
            i = 0
            while i < len(text_symbols) - 1:
                adjacent_symbols = text_symbols[i:i+2]
                adjacent_symbols_str = "".join(adjacent_symbols)
                if adjacent_symbols_str == merged_symbol:
                    new_text_symbols.append(merged_symbol)
                    i += 2
                else:
                    new_text_symbols.append(text_symbols[i])
                    i += 1
            if i == len(text_symbols) - 1:
                new_text_symbols.append(text_symbols[-1])
            text_symbols = new_text_symbols

            if len(set(text_symbols)) > self.vocab_size:
                break
        
        # print((text_symbols))
        for special_token in self.special_tokens:
            self.trie.add_node(parent_nodes=[], new_node=special_token)
        self.trie.mount_trie(set(text_symbols))
        self.token_to_idx = self.trie.token_to_idx
        self.idx_to_token = {v: k for k, v in self.token_to_idx.items()}
        print("Tokenizer treinado.")
        return

    def encode(self, token):
        # print(f"Vai appendar token {token}")
        return self.token_to_idx[token]
        
    def encode_text(self, text):
        idxs = []
        text_pos = 0
        window_size = 1
        while text_pos < len(text):
            # print(self.trie.get_trie())
            # print(text[text_pos:text_pos+1])
            paths, longest_text = self.trie.get_child_nodes(text[text_pos:text_pos+window_size])
            
            if paths == []:
                window_size += 1
                if text_pos + window_size > len(text):
                    idxs.append(self.unk_token_idx())
                    text_pos += 1
                continue
            window_size = 1

            longest_text_len = len(longest_text)
            text_slice = text[text_pos:text_pos+longest_text_len]

            if text_slice == longest_text:
                # print(f"1-APPENDOU {self.encode(text_slice)}")
                idxs.append(self.encode(text_slice))
                text_pos += longest_text_len
                continue

            nodes = []
            for path in paths:
                nodes.extend(path)
            nodes = sorted(set(nodes))
            
            selected_node_text = ""
            max_node_text_len = 0
            for node_text in nodes:
                node_text_len = len(node_text)
                text_slice = text[text_pos:text_pos+node_text_len]
                if text_slice == node_text and node_text_len >= max_node_text_len:
                    selected_node_text = node_text
                    max_node_text_len = node_text_len

            if selected_node_text == "":
                # print(f"2-APPENDOU {self.unk_token_idx()}")
                idxs.append(self.unk_token_idx())
                text_pos += 1

            # print(f"3-APPENDOU {self.encode(selected_node_text)}")
            idxs.append(self.encode(selected_node_text))
            text_pos += max_node_text_len

        return idxs
    
    def decode(self, token_idx):
        return self.idx_to_token[token_idx]
    
    def detect_special_token(self, str_):
        return True if str_[0:5] in self.special_tokens else False
    
    def sos_token(self):
        return '<SOS>'
    
    def sos_token_idx(self):
        return self.token_to_idx['<SOS>']

    def eos_token(self):
        return '<EOS>'
    
    def eos_token_idx(self):
        return self.token_to_idx['<EOS>']
    
    def pad_token(self):
        return '<PAD>'
    
    def pad_token_idx(self):
        return self.token_to_idx['<PAD>']
    
    def unk_token(self):
        return '<UNK>'
    
    def unk_token_idx(self):
        return self.token_to_idx['<UNK>']
    
    def eop_token(self):
        return '<EOP>'
    
    def eop_token_idx(self):
        return self.token_to_idx['<EOP>']
    
    def get_vocab_size(self):
        return self.vocab_size

In [31]:
tokenizer_bpe = TokenizerBPE()

In [32]:
# Exemplo de BPE de mercado com Hugging Face Tokenizers
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Texto de exemplo
corpus = ["Eita preula, esse eh o texto pra conseguir verificar textualmente se deu bom ou deu ruim essa questao do bom bom que mente. Mentemente mente."]

# Inicializa o tokenizer BPE
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=["<PAD>", "<SOS>", "<EOS>", "<UNK>", "<EOP>"], vocab_size=50)

# Treina o tokenizer
tokenizer.train_from_iterator(corpus, trainer)

# Mostra o vocabulário aprendido
print("VOCAB BPE MERCADO:")
for token, idx in tokenizer.get_vocab().items():
    print(f"{token}: {idx}")

# Tokeniza o texto de exemplo
output = tokenizer.encode(corpus[0])
print("Tokens:", output.tokens)
print("IDs:", output.ids)

VOCAB BPE MERCADO:
Mente: 47
x: 29
e: 13
xt: 44
q: 23
ui: 43
o: 21
E: 7
c: 11
u: 27
m: 19
i: 17
n: 20
al: 48
h: 16
s: 25
f: 14
v: 28
deu: 39
d: 12
t: 26
<EOP>: 4
M: 8
te: 30
Ei: 46
bo: 34
es: 35
en: 31
g: 15
eu: 36
l: 18
<EOS>: 2
r: 24
.: 6
ente: 32
qu: 41
<SOS>: 1
,: 5
a: 9
p: 22
mente: 33
pr: 40
<UNK>: 3
text: 45
ar: 49
<PAD>: 0
ta: 42
b: 10
se: 37
bom: 38
Tokens: ['Ei', 'ta', 'pr', 'eu', 'l', 'a', ',', 'es', 'se', 'e', 'h', 'o', 'text', 'o', 'pr', 'a', 'c', 'o', 'n', 'se', 'g', 'ui', 'r', 'v', 'e', 'r', 'i', 'f', 'i', 'c', 'ar', 'text', 'u', 'al', 'mente', 'se', 'deu', 'bom', 'o', 'u', 'deu', 'r', 'ui', 'm', 'es', 's', 'a', 'qu', 'es', 'ta', 'o', 'd', 'o', 'bom', 'bom', 'qu', 'e', 'mente', '.', 'Mente', 'mente', 'mente', '.']
IDs: [46, 42, 40, 36, 18, 9, 5, 35, 37, 13, 16, 21, 45, 21, 40, 9, 11, 21, 20, 37, 15, 43, 24, 28, 13, 24, 17, 14, 17, 11, 49, 45, 27, 48, 33, 37, 39, 38, 21, 27, 39, 24, 43, 19, 35, 25, 9, 41, 35, 42, 21, 12, 21, 38, 38, 41, 13, 33, 6, 47, 33, 33, 6]


In [33]:
tokenizer_bpe.fit("Eita preula, esse eh o texto pra conseguir verificar textualmente se deu bom ou deu ruim essa questao do bom bom que mente. Mentemente mente.")

Iniciando treinamento do TokenizerBPE...
Detectando e removendo tokens especiais...


100%|██████████| 137/137 [00:00<00:00, 136619.03it/s]


Removendo identificadores de tokens especiais...
Treinando BPE...


  0%|          | 23/10000 [00:00<00:00, 11416.45it/s]

Não existem símbolos possíveis de serem mergeados. Tokenizer treinado.
[' ', ',', 'E', 'M', 'a', 'a ', 'bom ', 'c', 'd', 'deu ', 'e', 'ente', 'es', 'eu', 'f', 'g', 'h', 'i', 'l', 'm ', 'mente ', 'mente.', 'n', 'o', 'o ', 'pr', 'qu', 'r', 'r ', 's', 'se', 'se ', 't', 'text', 'u', 'ui', 'v']
Tokenizer treinado.


In [34]:
tokenizer_bpe.token_to_idx

{'<PAD>': 0,
 '<SOS>': 1,
 '<EOS>': 2,
 '<UNK>': 3,
 '<EOP>': 4,
 ' ': 5,
 ',': 6,
 'E': 7,
 'M': 8,
 'a': 9,
 'a ': 10,
 'bom ': 11,
 'c': 12,
 'd': 13,
 'deu ': 14,
 'e': 15,
 'ente': 16,
 'es': 17,
 'eu': 18,
 'f': 19,
 'g': 20,
 'h': 21,
 'i': 22,
 'l': 23,
 'm ': 24,
 'mente ': 25,
 'mente.': 26,
 'n': 27,
 'o': 28,
 'o ': 29,
 'pr': 30,
 'qu': 31,
 'r': 32,
 'r ': 33,
 's': 34,
 'se': 35,
 'se ': 36,
 't': 37,
 'text': 38,
 'u': 39,
 'ui': 40,
 'v': 41}

In [35]:
tokenizer_bpe.trie.get_child_nodes("pr")

([['pr']], 'pr')

In [36]:
tokenizer_bpe.encode_text("Eita preula, esse eh o texto pra conseguir verificar textualmente se deu bom ou deu ruim essa questao do bom bom que mente. Mentemente mente.")

[7,
 22,
 37,
 10,
 30,
 18,
 23,
 9,
 6,
 5,
 17,
 36,
 15,
 21,
 5,
 29,
 38,
 29,
 30,
 10,
 12,
 28,
 27,
 35,
 20,
 40,
 33,
 41,
 15,
 32,
 22,
 19,
 22,
 12,
 9,
 33,
 38,
 39,
 9,
 23,
 25,
 36,
 14,
 11,
 28,
 39,
 5,
 14,
 32,
 40,
 24,
 17,
 34,
 10,
 31,
 17,
 37,
 9,
 29,
 13,
 29,
 11,
 11,
 31,
 15,
 5,
 26,
 5,
 8,
 16,
 25,
 26]

In [37]:
text_sample = "O rato roeu a roupa do rei de Roma. A rainha de Portugal gosta de queijo e pão. <SOS> Era uma vez um gato preto. <EOS>"
tokenizer_bpe = TokenizerBPE(vocab_size=50_000)
tokenizer_bpe.fit(text_sample)

Iniciando treinamento do TokenizerBPE...
Detectando e removendo tokens especiais...


100%|██████████| 114/114 [00:00<?, ?it/s]


Removendo identificadores de tokens especiais...
Treinando BPE...


  0%|          | 11/10000 [00:00<00:00, 10886.58it/s]

Não existem símbolos possíveis de serem mergeados. Tokenizer treinado.
[' ', ' r', ' ra', '. ', 'A', 'E', 'O', 'P', 'R', 'a', 'a ', 'a d', 'a de ', 'd', 'e', 'e ', 'ei', 'g', 'ga', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'to', 'u', 'um', 'v', 'z', 'ã']
Tokenizer treinado.


In [38]:
# Limpa o tokenizer para uso real.
tokenizer_bpe = TokenizerBPE()

### Dataset

#### Pre-training dataset

In [39]:
import json
from torch.utils.data import Dataset

In [40]:
dataset = pd.read_parquet('dataset_text/wiki_pt_0-50k.parquet')

In [41]:
dataset.head()

,id,url,title,text
0,220,https://pt.wikipedia.org/wiki/Astronomia,Astronomia,Astronomia é uma ciência natural que estuda co...
1,223,https://pt.wikipedia.org/wiki/Am%C3%A9rica%20L...,América Latina,A América Latina (; ) é uma região do continen...
2,224,https://pt.wikipedia.org/wiki/Albino%20Forjaz%...,Albino Forjaz de Sampaio,Albino Maria Pereira Forjaz de Sampaio (Lisboa...
3,226,https://pt.wikipedia.org/wiki/Anno%20Domini,Anno Domini,Anno Domini (A.D.) é uma expressão em latim qu...
4,228,https://pt.wikipedia.org/wiki/Aquiles,Aquiles,"Aquiles (), na mitologia grega, foi um herói d..."


In [42]:
np.random.seed(42)  # Para reprodutibilidade
dataset['split'] = np.random.choice(['train', 'test'], size=len(dataset), p=[0.99, 0.01])

In [43]:
dataset.head()

,id,url,title,text,split
0,220,https://pt.wikipedia.org/wiki/Astronomia,Astronomia,Astronomia é uma ciência natural que estuda co...,train
1,223,https://pt.wikipedia.org/wiki/Am%C3%A9rica%20L...,América Latina,A América Latina (; ) é uma região do continen...,train
2,224,https://pt.wikipedia.org/wiki/Albino%20Forjaz%...,Albino Forjaz de Sampaio,Albino Maria Pereira Forjaz de Sampaio (Lisboa...,train
3,226,https://pt.wikipedia.org/wiki/Anno%20Domini,Anno Domini,Anno Domini (A.D.) é uma expressão em latim qu...,train
4,228,https://pt.wikipedia.org/wiki/Aquiles,Aquiles,"Aquiles (), na mitologia grega, foi um herói d...",train


In [44]:
dataset.text[1].split('\n\n')[0][:2000]

'A América Latina (; ) é uma região do continente americano que engloba os países onde são faladas, primordialmente, línguas românicas (derivadas do latim) — no caso, o espanhol, o português e o francês — visto que, historicamente, a região foi maioritariamente dominada pelos impérios coloniais europeus Espanhol e Português. A América Latina tem uma área de cerca de  km², o equivalente a cerca de 3,9% da superfície da Terra (ou 14,1% de sua superfície emersa terrestre). Em 2008, a sua população foi estimada em mais de 569 milhões de pessoas. Os países do restante do continente americano tiveram uma colonização majoritariamente realizada por povos europeus de cultura anglo-saxônica ou neerlandesa (ver América Anglo-Saxônica). Vale ressaltar algumas exceções, como Québec, que não é um país independente, mas uma província de maioria francófona que pertence ao Canadá; o estado da Luisiana, que também foi colonizado por franceses, mas pertence aos Estados Unidos e os estados do sudoeste est

In [45]:
def df_wiki_prep(df_wiki, sequence_length=512, overlap=256, tokenizer=TokenizerChar()):
    text_items, ids, urls, titles, splits = [], [], [], [], []
    sos = tokenizer.sos_token()  # '<SOS>'
    eos = tokenizer.eos_token()  # '<EOS>'
    for instance_idx in tqdm(range(df_wiki.shape[0]), desc='Generating training instances...'):
        title = df_wiki.title[instance_idx]
        text = sos + df_wiki.text[instance_idx] + eos
        id = df_wiki.id[instance_idx]
        url = df_wiki.url[instance_idx]
        split = df_wiki.split[instance_idx]
        text_length = len(text)
        start = 0
        while start + sequence_length < text_length:
            sliced_text = f'Tema: {title}. Texto: ' + text[start:start + sequence_length]
            text_items.append(sliced_text)
            ids.append(id)
            urls.append(url)
            titles.append(title)
            splits.append(split)
            start += sequence_length - overlap
        if text_length - start >= 256:
            sliced_text = text[start:text_length]
        elif text_length > sequence_length:
            sliced_text = text[-256:]
        else:
            sliced_text = text
        text_items.append(sliced_text)
        ids.append(id)
        urls.append(url)
        titles.append(title)
        splits.append(split)
    df_wiki_preped = pd.DataFrame(
        {
            'id': ids,
            'urls': urls,
            'titles': titles,
            'split': splits,
            'text': text_items
        }
    )
    return df_wiki_preped

In [46]:
df_wiki_preped = df_wiki_prep(dataset, sequence_length=1024, overlap=256, tokenizer=tokenizer_bpe)

Generating training instances...: 100%|██████████| 50000/50000 [00:02<00:00, 24528.67it/s]


In [47]:
df_wiki_preped.text[0]

'Tema: Astronomia. Texto: <SOS>Astronomia é uma ciência natural que estuda corpos celestes (como estrelas, planetas, cometas, nebulosas, aglomerados de estrelas, galáxias) e fenômenos que se originam fora da atmosfera da Terra (como a radiação cósmica de fundo em micro-ondas). Preocupada com a evolução, a física e a química de objetos celestes, bem como a formação e o desenvolvimento do universo.\n\nA astronomia é uma das mais antigas ciências. Culturas pré-históricas deixaram registrados vários artefatos astronômicos, como Stonehenge, os montes de Newgrange e os menires. As primeiras civilizações, como os babilônios, gregos, chineses, indianos, persas e maias realizaram observações metódicas do céu noturno. No entanto, a invenção do telescópio permitiu o desenvolvimento da astronomia moderna. Historicamente, a astronomia incluiu disciplinas tão diversas como astrometria, navegação astronômica, astronomia observacional e a elaboração de calendários. Durante o período medieval, seu estu

In [48]:
df_wiki_preped.text[1]

'Tema: Astronomia. Texto: moderna. Historicamente, a astronomia incluiu disciplinas tão diversas como astrometria, navegação astronômica, astronomia observacional e a elaboração de calendários. Durante o período medieval, seu estudo era obrigatório e estava incluído no Quadrivium que, junto com o Trivium, compunha a metodologia de ensino das sete Artes liberais.\n\nDurante o século XX, o campo da astronomia profissional dividiu-se em dois ramos: a astronomia observacional e a astronomia teórica. A primeira está focada na aquisição de dados a partir da observação de objetos celestes, que são então analisados utilizando os princípios básicos da física. Já a segunda é orientada para o desenvolvimento de modelos analíticos que descrevem objetos e fenômenos astronômicos. Os dois campos se complementam, com a astronomia teórica procurando explicar os resultados observacionais, bem com as observações sendo usadas para confirmar (ou não) os resultados teóricos.\n\nOs astrônomos amadores têm co

In [49]:
df_wiki_preped[df_wiki_preped['split'] == 'train'].to_parquet('dataset_text/wiki_pt_0-50k_train.parquet')
df_wiki_preped[df_wiki_preped['split'] == 'test'].to_parquet('dataset_text/wiki_pt_0-50k_test.parquet')

In [50]:

fit_tokenizer = False
if fit_tokenizer:
    # tokenizer = TokenizerChar()
    import pandas as pd
    import numpy as np

    # Carrega a base
    df = pd.read_parquet('dataset_text/wiki_pt_0-50k.parquet')

    # Sorteia 50 índices aleatórios
    np.random.seed(42)
    sample_idxs = np.random.choice(df.index, size=50, replace=False)

    # Concatena os textos dos artigos sorteados
    sample_text = " ".join(df.loc[sample_idxs, "text"].tolist())

    tokenizer = TokenizerBPE()
    tokenizer.fit(sample_text, max_bpe_iter=100_000)

In [51]:
!pip install joblib


[notice] A new release of pip available: 22.2.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
import joblib

save = False
if save:
    joblib.dump(tokenizer, 'tokenizer_bpe.joblib')

In [53]:
tokenizer = joblib.load('tokenizer_bpe.joblib')

In [54]:
len(tokenizer.token_to_idx.keys())

9327

In [55]:
tokenizer.trie.get_child_nodes("<SOS>")

([['<SOS>']], '<SOS>')

In [56]:
tokenizer.encode_text('<SOS>')

[1]

In [57]:
tokenizer.token_to_idx['<SOS>']

1

In [58]:
tokenizer.sos_token_idx()

1

In [59]:
tokenizer.encode_text('<SOS> ')

[1, 77]

In [60]:
print([tokenizer.decode(t) for t in tokenizer.encode_text(df_wiki_preped.text[0])])

['Te', 'm', 'a: ', 'Ast', 'ron', 'omi', 'a. ', 'Text', 'o: ', '<SOS>', 'Ast', 'ron', 'omi', 'a é ', 'uma ', 'ciência ', 'natural ', 'que ', 'estud', 'a cor', 'pos', ' ', 'cel', 'estes ', '(', 'como ', 'estre', 'l', 'as, ', 'plan', 'et', 'as, ', 'com', 'et', 'as, ', 'ne', 'bu', 'lo', 's', 'as, ', 'ag', 'lo', 'mer', 'ados ', 'de ', 'estre', 'l', 'as, ', 'gal', 'á', 'x', 'i', 'as', ') e ', 'f', 'en', 'ôm', 'enos ', 'que se ', 'origin', 'am ', 'for', 'a da ', 'at', 'mos', 'fer', 'a da ', 'Terra ', '(', 'como a ', 'radi', 'ação ', 'c', 'ós', 'm', 'ica ', 'de ', 'fund', 'o em ', 'm', 'icr', 'o-', 'ond', 'as', '). ', 'Pre', 'ocup', 'ada ', 'com a ', 'evolu', 'ção, ', 'a f', 'ísi', 'ca ', 'e a ', 'qu', 'ím', 'ica ', 'de ', 'ob', 'jet', 'os c', 'ele', 'st', 'es, ', 'bem como ', 'a f', 'or', 'm', 'ação e ', 'o des', 'envolviment', 'o do ', 'univers', 'o.\n\nA ', 'astro', 'nom', 'ia ', 'é uma ', 'das mais ', 'antig', 'as ', 'ci', 'ênci', 'as. ', 'Cultur', 'as ', 'pré-', 'históric', 'as ', 'deix',

In [61]:
class DatasetWikipedia(Dataset):
    def __init__(self, data_path='dataset_text/wiki_pt_0-50k.parquet', sequence_length=512, split='train', tokenizer=TokenizerBPE()):
        self.dataset = pd.read_parquet(data_path).reset_index()  # load_dataset("wikimedia/wikipedia", "20231101.en", split=split)
        self.dataset = self.dataset.loc[self.dataset['split'] == split]
        self.sequence_length = sequence_length
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset.text[idx]
        current_sequence = self.tokenizer.encode_text(text)
        if len(current_sequence) < self.sequence_length + 1:
            current_sequence += [self.tokenizer.pad_token_idx()] * (self.sequence_length + 1 - len(current_sequence))
        else:
            current_sequence = current_sequence[:self.sequence_length + 1]
        x = torch.tensor(current_sequence[:-1])
        y = torch.tensor(current_sequence[1:])
        return x, y

In [62]:
dataset_train = DatasetWikipedia('dataset_text/wiki_pt_0-50k_train.parquet', sequence_length=1024, split='train')

In [63]:
dataloader_train = DataLoader(dataset_train, batch_size=4, shuffle=True)

In [64]:
dataloader_train.__len__()

101339

In [65]:
len(list(dataset_train.__getitem__(0)[0]))

1024

#### Fine-tuning dataset

In [66]:
class DatasetFinancial(Dataset):
    def __init__(self, data_path='dataset_text/train.json', sequence_length=4096):  # length que engloba todo o contexto (percentil 99)
        self.data_path = data_path
        self.sequence_length = sequence_length
        self.tokenizer = TokenizerChar()
        with open(self.data_path, encoding='utf-8') as f:
            lines = f.readlines()
        self.lines = [line for line in tqdm(lines) if line.isascii()]

    def __len__(self):
        return len(self.lines)

    def readline_as_dict(self, line_number):
        try:
            return json.loads(self.lines[line_number])
        except json.JSONDecodeError:
            return dict()

    def __getitem__(self, line_idx):
        line_dict = self.readline_as_dict(line_idx)
        if len(line_dict.keys()) == 0:
            x = torch.tensor([self.tokenizer.pad_token_idx()] * self.sequence_length)
            y = torch.tensor([self.tokenizer.pad_token_idx()] * self.sequence_length)
            return x, y
        system_text = line_dict['system']
        user_text = line_dict['user']
        assistant_text = line_dict['assistant'] 
        current_sequence = (
            [self.tokenizer.sos_token_idx()]
            + [self.tokenizer.encode(c) for c in system_text]
            + [self.tokenizer.encode(' ')]
            + [self.tokenizer.encode(c) for c in user_text]
            + [self.tokenizer.eop_token_idx()]
            + [self.tokenizer.encode(c) for c in assistant_text]
            + [self.tokenizer.eos_token_idx()]
        )
        if len(current_sequence) < self.sequence_length + 1:
            current_sequence += [self.tokenizer.pad_token_idx()] * (self.sequence_length + 1 - len(current_sequence))
        else:
            current_sequence = current_sequence[:self.sequence_length + 1]
        x = torch.tensor(current_sequence[:-1])
        y = torch.tensor(current_sequence[1:])
        return x, y

In [67]:
dataset_financial = DatasetFinancial()

100%|██████████| 518182/518182 [00:00<00:00, 2823057.70it/s]


In [68]:
dataset_financial[0]

(tensor([257,  10,  32,  ...,   0,   0,   0]),
 tensor([10, 32, 69,  ...,  0,  0,  0]))

In [69]:
len(dataset_financial)

442677

## Model Training

### Training Loop

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
writer = SummaryWriter(log_dir='runs/transformer_experiment_1024_tokens_v3_20250824')
sequence_length = 1024
batch_size = 2
accumulation_steps = 16
dataset_train = DatasetWikipedia('dataset_text/wiki_pt_0-50k_train.parquet', sequence_length=sequence_length, split='train', tokenizer=tokenizer)
dataset_test = DatasetWikipedia('dataset_text/wiki_pt_0-50k_test.parquet', sequence_length=sequence_length, split='test', tokenizer=tokenizer)
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(dataset_test, batch_size=batch_size, shuffle=True)
vocab_size = dataset_train.tokenizer.get_vocab_size()

d_model = 512
num_layers = 6
num_heads = 8
d_ff = 2048
dropout = 0.1
max_seq_length = sequence_length
# model = TransformerDecoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length)
model = TransformerDecoder(vocab_size, d_model, max_seq_length, num_layers, d_ff, num_heads, dropout=0.1)
model.to(device)
load_from_checkpoint = False

if load_from_checkpoint:
    checkpoint_path = 'model_checkpoints/model_checkpoint_4.pth'  # ajuste para o arquivo desejado
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))

tgt_mask = (1 - torch.triu(
  torch.ones(1, sequence_length, sequence_length), diagonal=1)
).bool()

def init_weights(module):
    if isinstance(module, (nn.Linear)):
        init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
        if module.bias is not None:
            init.zeros_(module.bias)
if not load_from_checkpoint:
    model.apply(init_weights)

optimizer = Adam(model.parameters(), lr=0.75*1e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)
n_epochs = 1
n_batches = int(dataset_train.__len__() // batch_size)

print("Starting model training...")
start_epoch = 0

optimizer.zero_grad()
for epoch in range(start_epoch, start_epoch + n_epochs):
    print(f"Epoch: {epoch + 1}")
    avg_loss = 0
    model.train()
    for batch_idx, batch in enumerate(tqdm(dataloader_train, total=n_batches)):
        x, y = batch
        x = x.to(device)
        y = y.to(device)
        outputs = model(x, tgt_mask.to(device))
        loss = loss_fn(outputs.view(-1, vocab_size), y.view(-1))
        avg_loss += loss.item()
        loss = loss / accumulation_steps
        loss.backward()

        if (batch_idx + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        writer.add_scalar('Loss/train', loss.item() * accumulation_steps, epoch * n_batches + batch_idx)

    torch.save(model.state_dict(), f'model_checkpoints/model_checkpoint_{epoch+1}.pth')

    avg_loss /= (batch_idx + 1)
    print(f"Average epoch training loss: {avg_loss}")
    print(f"Last batch training loss: {loss * accumulation_steps}")

    model.eval()
    avg_loss = 0
    for batch_idx, batch in enumerate(tqdm(dataloader_test, total=dataloader_test.__len__())):
        x, y = batch
        x = x.to(device)
        y = y.to(device)
        outputs = model(x, tgt_mask.to(device))
        loss = loss_fn(outputs.view(-1, vocab_size), y.view(-1))
        avg_loss += loss.item()
    
    avg_loss /= (batch_idx + 1)
    print(f"Epoch validation loss: {avg_loss}")
    writer.add_scalar('Loss/val', avg_loss, epoch)

writer.close()

cuda
Starting model training...
Epoch: 1


  6%|▌         | 11508/202676 [1:15:26<21:36:49,  2.46it/s]Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x0000026E7C569360>>
Traceback (most recent call last):
  File "c:\Users\Public\git\Estudos\transformer-fernando\env_gpu\lib\site-packages\ipykernel\ipkernel.py", line 794, in _clean_thread_parent_frames
    if phase != "start":
KeyboardInterrupt: 
  6%|▌         | 11534/202676 [1:15:37<23:31:43,  2.26it/s]

In [ ]:
def make_tgt_mask(sequence_length, device):
    tgt_mask = torch.tril(torch.ones(sequence_length, sequence_length, dtype=torch.bool)).to(device)
    return tgt_mask

In [ ]:
import torch

def predict_fernando(start_text, model, tokenizer, max_sequence_length=1024, temperature=1.0):
    model.eval()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    sequence = [tokenizer.sos_token_idx()] + [tokenizer.encode(c) for c in start_text]

    input_tokens = torch.tensor(sequence, dtype=torch.long).unsqueeze(0).to(device) # acrescenta dimensão batch_size=1
    current_text = start_text

    with torch.no_grad():
        for _ in range(len(start_text), max_sequence_length):
            outputs = model(input_tokens, tgt_mask=make_tgt_mask(input_tokens.shape[1], device))
            log_probs = outputs[0, -1] / temperature

            predicted_token_idx = torch.distributions.Categorical(logits=log_probs).sample().item()

            if predicted_token_idx == tokenizer.eos_token_idx():
                break

            current_text += tokenizer.decode(predicted_token_idx)
            input_tokens = torch.cat((input_tokens, torch.tensor([[predicted_token_idx]]).to(device)), dim=1)

    print('Texto predito:', current_text)
    return current_text


In [ ]:
predict_fernando(start_text="Tema: América. Texto: <SOS> A ", tokenizer=TokenizerChar(), model=model)

Texto predito: Tema: América. Texto: <SOS> A abriga animal é um acumulado de gênero significativo de chuva da província de Santa Cruz do Sul Su


'Tema: América. Texto: <SOS> A abriga animal é um acumulado de gênero significativo de chuva da província de Santa Cruz do Sul Su'

In [ ]:
predict_fernando(start_text="Tema: América. Texto: <SOS> A ", tokenizer=TokenizerChar(), model=model)

Texto predito: Tema: América. Texto: <SOS> A América (2 e 1) é um terminal com cerca de 1.420 metros acima do nível do mar. O território francê


'Tema: América. Texto: <SOS> A América (2 e 1) é um terminal com cerca de 1.420 metros acima do nível do mar. O território francê'

In [ ]:
predict_fernando(start_text="Alagoas", tokenizer=TokenizerChar(), model=model)

Texto predito: Alagoas (1986), são herbáceos (1980), que formam uma linguagem molecular: ela estava junto a recuperar na novela, a gradualma ao


'Alagoas (1986), são herbáceos (1980), que formam uma linguagem molecular: ela estava junto a recuperar na novela, a gradualma ao'

In [ ]:
predict_fernando(start_text="A América Latina é ", tokenizer=TokenizerChar(), model=model)

Texto predito: A América Latina é uma comuna italiana da região da Campania, província de Perna, com cerca 3.518 habitantes. Estende-se por uma área de 7 km², tendo uma densidade populacional de 44 hab/km². Faz fronteiro com Cagmasata, Campania, Castalno, Corliano, San Riosini, Rosaazle, São Rongellindo.

Demografia

Comunas de Perna (província)


'A América Latina é uma comuna italiana da região da Campania, província de Perna, com cerca 3.518 habitantes. Estende-se por uma área de 7 km², tendo uma densidade populacional de 44 hab/km². Faz fronteiro com Cagmasata, Campania, Castalno, Corliano, San Riosini, Rosaazle, São Rongellindo.\n\nDemografia\n\nComunas de Perna (província)'

In [ ]:
predict_fernando(start_text="Portugal é ", tokenizer=TokenizerChar(), model=model)

Texto predito: Portugal é um município brasileiro do estado de Galiza, situado na Região Paulista, Região Metropolitana do Rio Grande do Sul em Belo, Região Metropolitana, Região Metropolitana do Centro Estadual de Santa Cruz, Rio Grande do Sul. Recentemente em 1895, enquanto extinguiu ao Rio Grande do Sul, período do Maranhão. Conta-se em 1898, iniciou-se com o município pertencente à Região Metropolitana do Rio Grande do Sul, sendo a 2.ª cesquisa do município de Galiza, ao norte da Região Metropolitana do Sul e a mois censo de dois municípios.

Ver também 
 Lista de municípios de Galiza
 Caldas Amorais
 Hospitalismo da Rio Grande do Sul
 Luga de Suzana
 Lista de estilos do século I a.C.


'Portugal é um município brasileiro do estado de Galiza, situado na Região Paulista, Região Metropolitana do Rio Grande do Sul em Belo, Região Metropolitana, Região Metropolitana do Centro Estadual de Santa Cruz, Rio Grande do Sul. Recentemente em 1895, enquanto extinguiu ao Rio Grande do Sul, período do Maranhão. Conta-se em 1898, iniciou-se com o município pertencente à Região Metropolitana do Rio Grande do Sul, sendo a 2.ª cesquisa do município de Galiza, ao norte da Região Metropolitana do Sul e a mois censo de dois municípios.\n\nVer também \n Lista de municípios de Galiza\n Caldas Amorais\n Hospitalismo da Rio Grande do Sul\n Luga de Suzana\n Lista de estilos do século I a.C.'